In [ ]:
%pip install rouge_score bert_score evaluate
from datasets import load_dataset, Dataset
import pandas as pd
import numpy as np
import random as rd
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, Trainer, TrainingArguments, AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from tqdm import tqdm
import evaluate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from peft import LoraConfig, get_peft_model, TaskType
from evaluate import load
from sklearn.model_selection import train_test_split

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Set pad_token for batching
model = GPT2LMHeadModel.from_pretrained("gpt2").to("cuda")

In [ ]:
ds_train = load_dataset("FreedomIntelligence/RAG-Instruct", split="train[:80%]")
ds_test = load_dataset("FreedomIntelligence/RAG-Instruct", split="train[80%:]")
rankings_train = np.load('/kaggle/input/train-document-score/xenc_scores_train-stsb-distilroberta-base.npy')
rankings_test = np.load('/kaggle/input/test-score-npy/xenc_scores_test-stsb-distilroberta-base.npy')

k = 3

In [ ]:
def build_df(ds, doc_rankings):
    docs = [d for d in ds['documents']]
    questions = [q for q in ds['question']]
    answers = [a for a in ds['answer']]
    data = []

    for i, (q, a) in enumerate(zip(questions, answers)):
        ranked_indices = [int(t[1]) for t in doc_rankings[i][:k] if float(t[0]) > 1e-6]
        top_docs = [docs[i][idx] for idx in ranked_indices]
        data.append({
            'question': q,
            'answer': a,
            'topk_documents': top_docs
        })
        if(1 == 0):
            print(i)
            print("Question: " + q + "\n")
            print("Documents:\n")
            for d in top_docs:
                print(d + "\n")
            print("Answer: " + a + "\n")
    df = pd.DataFrame(data)
    topk_df = Dataset.from_pandas(df)
    return topk_df

In [ ]:
debug_range = range(0,1)
debug_dataset = ds_train[30:31]
debug_scores = rankings_train[30:31]
topk_ds_debug = build_df(debug_dataset, debug_scores)
debug_dataset_questions = ds_train['question'][30:31]
debug_dataset_answers = ds_train['answer'][30:31]
debug_dataset_documents = ds_train['documents'][30:31]
i = 0
print(str(i) + "\n")
print("Dataset Question:\n" + debug_dataset_questions[i] + "\n")
print("DF Question:\n" + topk_ds_debug[i]['question'] + "\n")
print("All Documents:\n")
for i,d in enumerate(debug_dataset_documents[i]):
    print(str(i) + ": " + d + "\n")
print("Scores:\n")
for s in debug_scores[0]:
    print(s)
print("\nTop K Documents:\n")
for d in topk_ds_debug[0]['topk_documents']:
    print(d + "\n")
print("Dataset Answer: " + debug_dataset_answers[0] + "\n")
print("DS Answer: " + topk_ds_debug[0]['answer'] + "\n")

In [ ]:
topk_ds_train = build_df(ds_train, rankings_train)
topk_ds_testval = build_df(ds_test, rankings_test)

topk_ds_split = topk_ds_testval.train_test_split(
    test_size=0.25,
    shuffle=True,
    seed=42
)

topk_ds_val = topk_ds_split["train"]
topk_ds_test = topk_ds_split["test"] 

### Performance Evaluation

In [ ]:
import torch
import math
import evaluate
from tqdm import tqdm
import re

def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)  # Collapse multiple spaces
    return text

def generate_answer(model, tokenizer, question, context, max_length=512):
    # Prompt in stile naturale adatto a GPT-2
    prompt = f"{context}\n\nQuestion: {question}\nAnswer:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            num_beams=4,
            early_stopping=True
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Estrai solo la parte dopo "Answer:"
    if "Answer:" in decoded:
        return decoded.split("Answer:")[-1].strip()
    else:
        return decoded.strip()

from sentence_transformers import SentenceTransformer, util
import torch
import evaluate
import math
from tqdm import tqdm

def evaluate_model(model, tokenizer, dataset, max_samples=5, max_new_tokens=50, print_examples=True):
    model.eval()
    dataset = dataset.select(range(min(len(dataset), max_samples)))

    bertscore = evaluate.load("bertscore")
    rouge = evaluate.load("rouge")
    em = evaluate.load("exact_match")

    embed_model = SentenceTransformer("all-MiniLM-L6-v2")

    predictions = []
    references = []
    losses = []
    faithfulness_scores = []
    relevance_scores = []

    for i, example in enumerate(tqdm(dataset, desc="Evaluating")):
        question = example["question"]
        context_list = example["topk_documents"]
        context = "\n".join(context_list)
        reference_text = example["answer"]

        # PPL
        full_target_text = f"{context}\n\nQuestion: {question}\nAnswer: {reference_text}"
        encoding = tokenizer(full_target_text, return_tensors="pt", truncation=True, max_length=512)
        input_ids = encoding["input_ids"].to(model.device)
        attention_mask = encoding["attention_mask"].to(model.device)
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
            losses.append(outputs.loss.item())

        # Generazione
        prediction = generate_answer(model, tokenizer, question, context, max_length=max_new_tokens)

        # Faithfulness: sim(doc, risposta)
        doc_emb = embed_model.encode(" ".join(context_list), convert_to_tensor=True)
        pred_emb = embed_model.encode(prediction, convert_to_tensor=True)
        faithfulness_scores.append(util.cos_sim(pred_emb, doc_emb).item())

        # Relevance: sim(domanda, risposta)
        question_emb = embed_model.encode(question, convert_to_tensor=True)
        relevance_scores.append(util.cos_sim(pred_emb, question_emb).item())

        predictions.append(prediction)
        references.append(reference_text)

        if print_examples:
            print(f"\n--- Example {i+1} ---")
            print(f"Question:\n{question}")
            print(f"Context:\n{context}")
            print(f"Prediction:\n{prediction}")
            print(f"Reference:\n{reference_text}")

    bertscore_f1 = bertscore.compute(predictions=predictions, references=references, lang="en")["f1"]
    rouge_score = rouge.compute(predictions=predictions, references=references)
    em_score = em.compute(predictions=predictions, references=references)["exact_match"]
    perplexity_score = math.exp(sum(losses) / len(losses)) if losses else float("inf")

    return {
        "RougeL": rouge_score["rougeL"],
        "EM": em_score,
        "BERTScore_F1": sum(bertscore_f1) / len(bertscore_f1),
        "Perplexity": perplexity_score,
        "Faithfulness": sum(faithfulness_scores) / len(faithfulness_scores),
        "Relevance": sum(relevance_scores) / len(relevance_scores)
    }


In [ ]:
base_metrics = evaluate_model(
    model=model,
    tokenizer=tokenizer,
    dataset=topk_ds_test,
    max_samples= topk_ds_test.shape[0],
    print_examples=False
)

print("\nMetrics:")
for k, v in base_metrics.items():
    print(f"{k}: {v:.4f}")

### LoRA FT

In [ ]:
from itertools import chain

def preprocess_function(examples, tokenizer):
    inputs = []
    for question, docs, answer in zip(examples["question"], examples["topk_documents"], examples["answer"]):
        context = " ".join(chain.from_iterable(docs)) if isinstance(docs[0], list) else " ".join(docs)
        if isinstance(answer, list):
            answer = " ".join(map(str, answer))
        else:
            answer = str(answer)

        prompt = f"### Question:\n{question}\n\n### Context:\n{context}\n\n### Answer:\n"
        input_text = prompt + answer
        inputs.append(input_text)

    tokenized = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

def create_lora_model(model):
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["c_attn"],  # <- For GPT-2 attention layer
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    return get_peft_model(model, lora_config)


In [ ]:
def pd_to_hf_ds(dataset, tokenizer):
    if isinstance(dataset, pd.DataFrame):
        dataset = Dataset.from_pandas(dataset)
    
    tokenized_dataset = dataset.map(lambda x: preprocess_function(x, tokenizer), batched=True)
    return tokenized_dataset

In [ ]:
model = create_lora_model(model)
model.print_trainable_parameters()

topk_hf_train = pd_to_hf_ds(topk_ds_train, tokenizer)
topk_hf_val = pd_to_hf_ds(topk_ds_val, tokenizer)

training_args = TrainingArguments(
    output_dir="/kaggle/working/train",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    fp16=True,
    gradient_accumulation_steps=4,
    report_to="tensorboard",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=topk_hf_train,
    eval_dataset=topk_hf_val,
    tokenizer=tokenizer,
)

In [ ]:
train_result = trainer.train()

model.save_pretrained("gpt2-qa-1")
tokenizer.save_pretrained("gpt2-qa-1")

final_loss = train_result.training_loss
with open("gpt2-qa-1/final_loss.txt", "w") as f:
    f.write(str(final_loss))

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from peft import PeftModel

adapter_path = "/kaggle/working/gpt2-qa-1"
base_model_id = "gpt2"  # <-- imposta qui il modello base usato

# Carica il modello base
base_model = GPT2LMHeadModel.from_pretrained(base_model_id)

# Applica l'adapter
model = PeftModel.from_pretrained(base_model, adapter_path).to("cuda")

# Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(base_model_id)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id



ft_metrics = evaluate_model(
    model=model,
    tokenizer=tokenizer,
    dataset=topk_ds_test,
    max_samples= 10, #topk_ds_test.shape[0],
    print_examples=True
)

#Finetuned model
#Metrics:
#RougeL: 0.1023
#EM: 0.0000
#BERTScore_F1: 0.8049
#Perplexity: 46.3106

print("\nMetrics:")
for k, v in ft_metrics.items():
    print(f"{k}: {v:.4f}")

# Finetuned model
Metrics:
RougeL: 0.1481
EM: 0.0000
BERTScore_F1: 0.8372
Perplexity: 28.9390

# Base model
Metrics:
RougeL: 0.1461
EM: 0.0000
BERTScore_F1: 0.8354
Perplexity: 32.4523

Faithfullness and Relevance, Bleu, Token Lenght Ratio, Diversity